<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Implement Agent Logging </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Exercise: Agent Observability and Monitoring**



### Exercise
In this exercise you will **add a structured logging system** to an existing Travel Planning Agent built with LangGraph.

The agent is already written and working — your job is to instrument it so that every reasoning step, every tool call, and every final result is captured in a structured log.

By the end you will have:
- A `AgentLogger` class that records structured events
- Logging hooks injected into the LangGraph agent nodes
- A printed log report showing the full execution trail



### Scenario: Travel Planning Assistant
A LangGraph agent that helps users plan trips by searching destinations, checking flight prices, finding hotels, and building itineraries.  
You will add logging so every decision and tool call is auditable.




### Exercise Tasks


**Task 1** Define log event data structures

**Task 2** Implement the `AgentLogger`

**Task 3** Add logging to the agent reasoning node

**Task 4** Add logging to the tool execution node

**Task 5** Log the final result and print a report



## 1. Setup

In [1]:
!pip install -q langchain langchain-openai langgraph --quiet

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 13.6 MB/s eta 0:00:00
Enter your OpenAI API key: ··········
Setup complete


In [2]:
import time
import uuid
from datetime import datetime
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Imports ready")

Imports ready


## 2. The Travel Planning Tools (Pre-Built — Read Only)

The four tools below are already implemented for you.  
**Do not modify this.**

| Tool | What it does |
|------|-------------|
| `search_destinations` | Returns destination suggestions for a travel query |
| `check_flight_prices` | Returns mock round-trip flight prices |
| `get_hotel_recommendations` | Returns hotel options for a city and budget |
| `create_itinerary` | Generates a day-by-day itinerary outline |

Read through the tools so you understand what inputs and outputs to expect when you log them in Task 4.

In [3]:
import random

DESTINATIONS_DB = {
    "beach":    ["Bali, Indonesia", "Santorini, Greece", "Cancún, Mexico", "Maldives"],
    "mountain": ["Swiss Alps, Switzerland", "Banff, Canada", "Patagonia, Argentina"],
    "city":     ["Tokyo, Japan", "Paris, France", "New York, USA", "Istanbul, Turkey"],
    "culture":  ["Kyoto, Japan", "Rome, Italy", "Marrakech, Morocco", "Cairo, Egypt"],
    "adventure":["New Zealand", "Costa Rica", "Iceland", "Nepal"],
}

FLIGHT_PRICES = {
    "bali":        {"economy": 780,  "business": 2100},
    "santorini":   {"economy": 620,  "business": 1800},
    "cancún":      {"economy": 430,  "business": 1200},
    "tokyo":       {"economy": 890,  "business": 2400},
    "paris":       {"economy": 540,  "business": 1650},
    "swiss alps":  {"economy": 570,  "business": 1750},
    "iceland":     {"economy": 490,  "business": 1400},
    "new zealand": {"economy": 1100, "business": 2900},
}

@tool
def search_destinations(query: str) -> str:
    """Search for travel destinations based on a travel style or preference keyword."""
    time.sleep(0.06)
    q = query.lower()
    results = []
    for category, places in DESTINATIONS_DB.items():
        if category in q or q in category:
            results.extend(places)
    if not results:
        results = DESTINATIONS_DB["city"]
    return "Suggested destinations:\n" + "\n".join(f"• {p}" for p in results[:4])


@tool
def check_flight_prices(destination: str, travel_class: str = "economy") -> str:
    """Check round-trip flight prices for a destination. travel_class: 'economy' or 'business'."""
    time.sleep(0.05)
    dest_key = destination.lower().split(",")[0].strip()
    travel_class = travel_class.lower()
    if dest_key in FLIGHT_PRICES:
        price = FLIGHT_PRICES[dest_key].get(travel_class, FLIGHT_PRICES[dest_key]["economy"])
        return f"Round-trip flight to {destination} ({travel_class}): ${price:,} per person"
    base = 600 if travel_class == "economy" else 1800
    price = base + random.randint(-100, 300)
    return f"Round-trip flight to {destination} ({travel_class}): ~${price:,} per person (estimated)"


@tool
def get_hotel_recommendations(city: str, budget: str = "mid-range") -> str:
    """Get hotel recommendations for a city. budget: 'budget', 'mid-range', or 'luxury'."""
    time.sleep(0.07)
    hotels = {
        "budget":    [("Cozy Hostel Central", "$35/night"), ("City Budget Inn", "$55/night")],
        "mid-range": [("Comfort Suites Downtown", "$120/night"), ("Grand Plaza Hotel", "$155/night")],
        "luxury":    [("The Royal Palace Hotel", "$380/night"), ("Prestige Grand Resort", "$520/night")],
    }
    options = hotels.get(budget.lower(), hotels["mid-range"])
    lines = [f"• {name} — {price}" for name, price in options]
    return f"Hotels in {city} ({budget}):\n" + "\n".join(lines)


@tool
def create_itinerary(destination: str, days: int, interests: str) -> str:
    """Create a day-by-day travel itinerary for a destination."""
    time.sleep(0.04)
    activities = {
        "culture":  ["Visit local museums", "Explore old town", "Attend a cultural show"],
        "food":     ["Street food tour", "Cooking class", "Fine dining experience"],
        "nature":   ["Hiking trail", "Scenic viewpoint", "National park visit"],
        "adventure":["Zip-lining", "Scuba diving", "Rock climbing"],
        "relax":    ["Spa day", "Beach lounging", "Sunset cruise"],
    }
    interest_key = interests.lower().split()[0] if interests else "culture"
    acts = activities.get(interest_key, activities["culture"])
    itinerary = [f"Day {i+1}: {acts[i % len(acts)]} in {destination}" for i in range(days)]
    return f"{days}-Day Itinerary for {destination}:\n" + "\n".join(itinerary)


tools = [search_destinations, check_flight_prices, get_hotel_recommendations, create_itinerary]
print(f"{len(tools)} travel tools ready")

4 travel tools ready


### Task 1 — Define the `LogEvent` dataclass
A `LogEvent` captures a single moment in the agent's execution.

**Every log event must record:**
| Field | Type | Description |
|-------|------|-------------|
| `event_id` | `str` | A short unique ID (first 6 chars of a UUID) |
| `event_type` | `str` | One of: `"reasoning"`, `"tool_call"`, `"tool_result"`, `"final_result"` |
| `timestamp` | `str` | UTC timestamp as ISO string |
| `content` | `Any` | The actual data being logged (text, dict, etc.) |
| `latency_ms` | `float` | How long this step took in milliseconds (default `0.0`) |
| `metadata` | `Dict` | Any extra key/value pairs (default empty dict) |

---

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 1 — Define the LogEvent dataclass
# ─────────────────────────────────────────────────────────────────────────────
# Instructions:
#   • Use @dataclass decorator
#   • Include all 6 fields listed in the instructions above
#   • latency_ms should default to 0.0
#   • metadata should default to an empty dict (use field(default_factory=dict))

# TODO: Define the LogEvent dataclass here
import dataclasses

# ─────────────────────────────────────────────────────────────────────────────
# TASK 1 SOLUTION — LogEvent dataclass
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class LogEvent:
    """A single structured log event from one step of agent execution."""
    event_id:   str
    event_type: str          # reasoning | tool_call | tool_result | final_result
    timestamp:  str
    content:    Any
    latency_ms: float                      = 0.0
    metadata:   Dict[str, Any]             = field(default_factory=dict)

### Task 2 — Implement the `AgentLogger` class
The `AgentLogger` manages all log events for one agent session.

**It must support these methods:**

| Method | What it does |
|--------|-------------|
| `__init__` | Initialises an empty `events` list and a `session_id` |
| `log(event_type, content, latency_ms, metadata)` | Creates a `LogEvent`, appends it to `events`, prints a one-line summary to the console |
| `print_report()` | Prints a formatted report of ALL events in sequence |
| `to_dict()` | Returns a list of dicts (one per event) — useful for export |

**Hint for `log()`:** The one-line console print should show the event type icon, event type label, and a short preview of the content. Use a dict like `{"reasoning": "🧠", "tool_call": "🔧", "tool_result": "📦", "final_result": "✅"}` for icons.

**Hint for `print_report()`:** Loop through `self.events` and print each event's index, type, timestamp, latency, and content preview on one line.

In [5]:

# ─────────────────────────────────────────────────────────────────────────────
# TASK 2 — Implement the AgentLogger class
# ─────────────────────────────────────────────────────────────────────────────
# Instructions:
#   • __init__: create self.events = [] and self.session_id = short UUID
#   • log(): create a LogEvent, append it, print a one-line console summary
#   • print_report(): print all events with index, type, latency, content preview
#   • to_dict(): return [dataclasses.asdict(e) for e in self.events]

# TODO: Implement the AgentLogger class here
# ─────────────────────────────────────────────────────────────────────────────
# TASK 2 SOLUTION — AgentLogger class
# ─────────────────────────────────────────────────────────────────────────────
class AgentLogger:
    """Collects and reports all log events for one agent session."""

    ICONS = {
        "reasoning":    "🧠",
        "tool_call":    "🔧",
        "tool_result":  "📦",
        "final_result": "✅",
        "tool_error":   "❌",
    }

    def __init__(self):
        self.events:    List[LogEvent] = []
        self.session_id: str = str(uuid.uuid4())[:8]

    def log(self,
            event_type: str,
            content: Any,
            latency_ms: float = 0.0,
            metadata: Dict[str, Any] = None) -> LogEvent:
        """
        Create a LogEvent, store it, and print a one-line console summary.
        Returns the created LogEvent.
        """
        event = LogEvent(
            event_id   = str(uuid.uuid4())[:6],
            event_type = event_type,
            timestamp  = datetime.utcnow().isoformat(),
            content    = content,
            latency_ms = latency_ms,
            metadata   = metadata or {},
        )
        self.events.append(event)

        # One-line console print
        icon    = self.ICONS.get(event_type, "•")
        preview = str(content)[:100]
        lat_str = f" {latency_ms:.0f}ms" if latency_ms > 0 else ""
        print(f"  {icon} [{event_type.upper():13s}] {preview}{lat_str}")

        return event

    def print_report(self):
        """Print a formatted, numbered report of all logged events."""
        SEP = "─" * 65
        print(f"\n{SEP}")
        print(f"AGENT LOG REPORT  |  Session: {self.session_id}")
        print(f"  Total events: {len(self.events)}")
        print(SEP)
        for i, e in enumerate(self.events, 1):
            icon    = self.ICONS.get(e.event_type, "•")
            preview = str(e.content)[:80]
            lat_str = f"{e.latency_ms:6.0f}ms" if e.latency_ms else "      —"
            print(f"  {i:02d}. {icon} {e.event_type.upper():13s} | "
                  f"{lat_str} | {preview}")
        print(f"{SEP}\n")

    def to_dict(self) -> List[Dict]:
        """Serialise all events to a list of dicts (for JSON export)."""
        return [dataclasses.asdict(e) for e in self.events]

In [6]:
logger = AgentLogger()
print(f"AgentLogger ready — session: {logger.session_id}")

AgentLogger ready — session: 7ae5f962


### Quick Check — Test your `AgentLogger` manually

Before wiring the logger into the agent, verify it works on its own.  
Run the cell below. You should see three console lines (one per `log()` call) and then the full report with 3 events.

In [7]:
# Scheck — do not modify
test_logger = AgentLogger()

test_logger.log("reasoning",   "User wants a beach holiday in Southeast Asia")
test_logger.log("tool_call",   {"tool": "search_destinations", "args": {"query": "beach"}},
                latency_ms=62.4)
test_logger.log("tool_result", "Suggested destinations: Bali, Maldives, Cancún",
                latency_ms=62.4)

print()
test_logger.print_report()

# Verify structure
assert len(test_logger.events) == 3,            "Should have 3 events"
assert test_logger.events[0].event_type == "reasoning", "First event should be 'reasoning'"
assert isinstance(test_logger.to_dict(), list),  "to_dict() should return a list"
print("All sanity checks passed")

  🧠 [REASONING    ] User wants a beach holiday in Southeast Asia
  🔧 [TOOL_CALL    ] {'tool': 'search_destinations', 'args': {'query': 'beach'}} 62ms
  📦 [TOOL_RESULT  ] Suggested destinations: Bali, Maldives, Cancún 62ms


─────────────────────────────────────────────────────────────────
AGENT LOG REPORT  |  Session: a61a1b3d
  Total events: 3
─────────────────────────────────────────────────────────────────
  01. 🧠 REASONING     |       — | User wants a beach holiday in Southeast Asia
  02. 🔧 TOOL_CALL     |     62ms | {'tool': 'search_destinations', 'args': {'query': 'beach'}}
  03. 📦 TOOL_RESULT   |     62ms | Suggested destinations: Bali, Maldives, Cancún
─────────────────────────────────────────────────────────────────

All sanity checks passed


/tmp/ipykernel_2239/2870158388.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp  = datetime.utcnow().isoformat(),


## 4. The LangGraph Agent — Skeleton

The agent state and routing logic are provided below.  
**Do not modify this cell.**  

Here you will create the *instrumented* versions of `agent_node` and `tool_node` by adding logging calls.  
Reading this carefully will help you understand what data is available when you write those logging hooks.

In [8]:
# Agent state
class TravelState(TypedDict):
    messages: Annotated[list, add_messages]

# LLM bound to tools
llm_with_tools = llm.bind_tools(tools)

# Base tool node (no logging yet)
base_tool_node = ToolNode(tools)

# Routing function
def should_continue(state: TravelState) -> str:
    """Route to tools if the LLM requested a tool call, otherwise end."""
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return END

print("Agent skeleton loaded")
print("Graph topology:  START → agent_node → (tool_calls?) → tool_node → agent_node → END")

Agent skeleton loaded
Graph topology:  START → agent_node → (tool_calls?) → tool_node → agent_node → END


### Task 3 — Instrument the Agent Reasoning Node

Write `logged_agent_node(state)` — a wrapper around the LLM call that:

1. Records the **start time** before calling `llm_with_tools.invoke()`
2. Calls `llm_with_tools.invoke(state["messages"])` to get the response
3. Calculates `latency_ms` from start to finish
4. Calls `logger.log()` with:
   - `event_type = "reasoning"`
   - `content` = the response content text (use `response.content` — may be empty string if only tool calls)
   - `latency_ms` = the latency you calculated
   - `metadata` = `{"has_tool_calls": True/False}` depending on whether the response has tool calls
5. If the response **has tool calls**, also log each one with `event_type = "tool_call"` and `content` = a dict with `"tool_name"` and `"tool_args"` keys
6. Returns `{"messages": [response]}`

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 3 — Logged agent reasoning node
# ─────────────────────────────────────────────────────────────────────────────
# Instructions:
#   1. Record start time
#   2. Call llm_with_tools.invoke()
#   3. Calculate latency_ms
#   4. Log the reasoning step (event_type="reasoning")
#   5. If tool calls exist, also log each one (event_type="tool_call")
#   6. Return {"messages": [response]}

# ─────────────────────────────────────────────────────────────────────────────
# TASK 3 SOLUTION — Logged agent reasoning node
# ─────────────────────────────────────────────────────────────────────────────
def logged_agent_node(state: TravelState) -> TravelState:
    """LLM call with reasoning and tool-decision logging."""

    # 1. Record start time
    t0 = time.perf_counter()

    # 2. Call the LLM
    response = llm_with_tools.invoke(state["messages"])

    # 3. Calculate latency
    latency_ms = (time.perf_counter() - t0) * 1000

    has_tool_calls = bool(getattr(response, "tool_calls", None))

    # 4. Log the reasoning step
    logger.log(
        event_type = "reasoning",
        content    = response.content if response.content else "(deciding tool calls)",
        latency_ms = latency_ms,
        metadata   = {"has_tool_calls": has_tool_calls},
    )

    # 5. Log each tool call decision individually
    if has_tool_calls:
        for tc in response.tool_calls:
            logger.log(
                event_type = "tool_call",
                content    = {"tool_name": tc["name"], "tool_args": tc["args"]},
                metadata   = {"tool_call_id": tc.get("id", "")},
            )

    # 6. Return updated state
    return {"messages": [response]}

### Task 4 — Instrument the Tool Execution Node

Write `logged_tool_node(state)` — a wrapper around `base_tool_node` that:

1. Records the **start time** before calling `base_tool_node.invoke(state)`
2. Calls `base_tool_node.invoke(state)` to run the actual tools
3. Calculates `latency_ms` from start to finish
4. Loops through the result messages — for each `ToolMessage` found, call `logger.log()` with:
   - `event_type = "tool_result"`
   - `content` = the tool message content (the tool's return value)
   - `latency_ms` = the latency you calculated
   - `metadata` = `{"tool_call_id": msg.tool_call_id}`
5. Returns the result from `base_tool_node`

**Hint:** Check if a message is a `ToolMessage` with `isinstance(msg, ToolMessage)`  
**Hint:** Access tool call details from `last_msg = state["messages"][-1]` and `last_msg.tool_calls`

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 4 — Logged tool execution node
# ─────────────────────────────────────────────────────────────────────────────
# Instructions (see Section 5 markdown above for full spec):
#   1. Record start time
#   2. Call base_tool_node.invoke(state)
#   3. Calculate latency_ms
#   4. For each ToolMessage in results, log it (event_type="tool_result")
#   5. Return the base_tool_node result

# ─────────────────────────────────────────────────────────────────────────────
# TASK 4 SOLUTION — Logged tool execution node
# ─────────────────────────────────────────────────────────────────────────────
def logged_tool_node(state: TravelState) -> TravelState:
    """Tool execution with result logging."""

    # 1. Record start time
    t0 = time.perf_counter()

    # 2. Execute tools via the base ToolNode
    result = base_tool_node.invoke(state)

    # 3. Calculate latency
    latency_ms = (time.perf_counter() - t0) * 1000

    # 4. Log each tool result message
    for msg in result.get("messages", []):
        if isinstance(msg, ToolMessage):
            logger.log(
                event_type = "tool_result",
                content    = msg.content,
                latency_ms = latency_ms,
                metadata   = {"tool_call_id": msg.tool_call_id},
            )

    # 5. Return the result unchanged
    return result


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Build the graph using your logged nodes
# ─────────────────────────────────────────────────────────────────────────────
graph = StateGraph(TravelState)
graph.add_node("agent", logged_agent_node)
graph.add_node("tools", logged_tool_node)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")
agent = graph.compile()

print("Instrumented agent compiled")

Instrumented agent compiled


### Task 5 — `run_with_logging()`

Write a function `run_with_logging(query: str) -> str` that:

1. Resets the global logger to a fresh `AgentLogger()` instance (so each run gets a clean log)
2. Prints a header line: `"\n=== Agent Run: {query[:60]} ===\n"`
3. Invokes the agent: `agent.invoke({"messages": [HumanMessage(content=query)]})`
4. Extracts the final answer from `final_state["messages"][-1].content`
5. Calls `logger.log()` with:
   - `event_type = "final_result"`
   - `content` = the final answer string
   - `metadata` = `{"query": query, "total_events": len(logger.events)}`
6. Calls `logger.print_report()` to display the full log
7. Returns the final answer string

**Hint:** Remember to assign to the global `logger` variable — use `global logger` at the top of the function.

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 5 — run_with_logging()
# ─────────────────────────────────────────────────────────────────────────────
# Instructions (see Section 6 markdown above for full spec):
#   1. global logger; logger = AgentLogger()
#   2. Print run header
#   3. Invoke the agent
#   4. Extract final answer
#   5. Log the final_result event
#   6. Call logger.print_report()
#   7. Return final answer

# ─────────────────────────────────────────────────────────────────────────────
# TASK 5 SOLUTION — run_with_logging()
# ─────────────────────────────────────────────────────────────────────────────
def run_with_logging(query: str) -> str:
    """Run the agent for one query with full logging enabled."""

    # 1. Reset logger for a clean session
    global logger
    logger = AgentLogger()

    # 2. Print run header
    print(f"\n{'='*65}")
    print(f"  Agent Run: {query[:60]}")
    print(f"{'='*65}\n")

    # 3. Invoke the agent
    final_state = agent.invoke({"messages": [HumanMessage(content=query)]})

    # 4. Extract final answer
    final_answer = final_state["messages"][-1].content

    # 5. Log the final result
    logger.log(
        event_type = "final_result",
        content    = final_answer,
        metadata   = {
            "query":        query,
            "total_events": len(logger.events),
        },
    )

    # 6. Print the full log report
    logger.print_report()

    # 7. Return the answer
    return final_answer


print("run_with_logging() defined")

run_with_logging() defined


## Test Your Logging Implementation

Run each cell below to test your implementation end-to-end.  
For each query, you should see:
- Live console lines as the agent runs (from your `log()` method)
- A full structured report after each run (from `print_report()`)

If anything is missing from the log, go back and check Tasks 3–5.

In [13]:
#  Test 1: Simple query (should trigger 1–2 tool calls)
answer = run_with_logging(
    "I want to visit a beach destination. What do you suggest?"
)
print("\nFinal Answer:", answer[:200])


  Agent Run: I want to visit a beach destination. What do you suggest?



/tmp/ipykernel_2239/2870158388.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp  = datetime.utcnow().isoformat(),


  🧠 [REASONING    ] (deciding tool calls) 1638ms
  🔧 [TOOL_CALL    ] {'tool_name': 'search_destinations', 'tool_args': {'query': 'beach'}}
  📦 [TOOL_RESULT  ] Suggested destinations:
• Bali, Indonesia
• Santorini, Greece
• Cancún, Mexico
• Maldives 62ms
  🧠 [REASONING    ] Here are some great beach destinations you might consider:

1. **Bali, Indonesia** - Known for its s 1655ms
  ✅ [FINAL_RESULT ] Here are some great beach destinations you might consider:

1. **Bali, Indonesia** - Known for its s

─────────────────────────────────────────────────────────────────
AGENT LOG REPORT  |  Session: e470249d
  Total events: 5
─────────────────────────────────────────────────────────────────
  01. 🧠 REASONING     |   1638ms | (deciding tool calls)
  02. 🔧 TOOL_CALL     |       — | {'tool_name': 'search_destinations', 'tool_args': {'query': 'beach'}}
  03. 📦 TOOL_RESULT   |     62ms | Suggested destinations:
• Bali, Indonesia
• Santorini, Greece
• Cancún, Mexico
•
  04. 🧠 REASONING     |   1655

/tmp/ipykernel_2239/2870158388.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp  = datetime.utcnow().isoformat(),


In [14]:
#  Test 2: Multi-step query (should trigger 3+ tool calls)
answer = run_with_logging(
    "Plan a 4-day culture trip to Japan: search destinations, check economy "
    "flight prices to Tokyo, find mid-range hotels, and create an itinerary."
)
print("\nFinal Answer:", answer[:200])


  Agent Run: Plan a 4-day culture trip to Japan: search destinations, che



/tmp/ipykernel_2239/2870158388.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp  = datetime.utcnow().isoformat(),


  🧠 [REASONING    ] (deciding tool calls) 5057ms
  🔧 [TOOL_CALL    ] {'tool_name': 'search_destinations', 'tool_args': {'query': 'cultural trip to Japan'}}
  🔧 [TOOL_CALL    ] {'tool_name': 'check_flight_prices', 'tool_args': {'destination': 'Tokyo', 'travel_class': 'economy'
  🔧 [TOOL_CALL    ] {'tool_name': 'get_hotel_recommendations', 'tool_args': {'city': 'Tokyo', 'budget': 'mid-range'}}
  🔧 [TOOL_CALL    ] {'tool_name': 'create_itinerary', 'tool_args': {'destination': 'Tokyo', 'days': 4, 'interests': 'cul
  📦 [TOOL_RESULT  ] Suggested destinations:
• Tokyo, Japan
• Paris, France
• New York, USA
• Istanbul, Turkey 85ms
  📦 [TOOL_RESULT  ] Round-trip flight to Tokyo (economy): $890 per person 85ms
  📦 [TOOL_RESULT  ] Hotels in Tokyo (mid-range):
• Comfort Suites Downtown — $120/night
• Grand Plaza Hotel — $155/night 85ms
  📦 [TOOL_RESULT  ] 4-Day Itinerary for Tokyo:
Day 1: Visit local museums in Tokyo
Day 2: Explore old town in Tokyo
Day  85ms
  🧠 [REASONING    ] ### Cultural Trip 

/tmp/ipykernel_2239/2870158388.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp  = datetime.utcnow().isoformat(),


In [15]:
#  Validation — run this to check your implementation
print("Running validation checks...\n")

errors = []

# Check 1: logger has events
if len(logger.events) == 0:
    errors.append("No events logged — check your log() calls in Tasks 3–5")
else:
    print(f"Events logged: {len(logger.events)}")

# Check 2: all four event types are present
logged_types = {e.event_type for e in logger.events}
for expected_type in ["reasoning", "tool_call", "tool_result", "final_result"]:
    if expected_type in logged_types:
        print(f"Event type present: '{expected_type}'")
    else:
        errors.append(f"Missing event type: '{expected_type}'")

# Check 3: tool_call events have tool_name in content
tc_events = [e for e in logger.events if e.event_type == "tool_call"]
if tc_events:
    for e in tc_events:
        if isinstance(e.content, dict) and "tool_name" in e.content:
            print(f"tool_call content has 'tool_name' key")
            break
    else:
        errors.append("tool_call events missing 'tool_name' in content dict")

# Check 4: tool_result events have latency > 0
tr_events = [e for e in logger.events if e.event_type == "tool_result"]
if any(e.latency_ms > 0 for e in tr_events):
    print("tool_result events have latency_ms > 0")
else:
    errors.append("tool_result events have latency_ms = 0 — did you measure timing?")

# Check 5: final_result logged
fr_events = [e for e in logger.events if e.event_type == "final_result"]
if fr_events:
    print("final_result event present")
else:
    errors.append("No final_result event — check Task 5")

# Summary
print()
if errors:
    print("Issues found:")
    for err in errors:
        print(" ", err)
else:
    print("All checks passed! Your logging implementation is complete.")

Running validation checks...

Events logged: 11
Event type present: 'reasoning'
Event type present: 'tool_call'
Event type present: 'tool_result'
Event type present: 'final_result'
tool_call content has 'tool_name' key
tool_result events have latency_ms > 0
final_result event present

All checks passed! Your logging implementation is complete.
